# Structured chain of thought on 200 questions: a pilot

A small run of the structured-CoT experiment (`README.md` in this folder), to see whether reasoning step by step
helps Qwen3.5-2B use the gold sheet before spending about 15 GPU hours on all 2,728 questions. The questions are
explore200's (`experiments/2026-11-18-scene-sheet-formats/explore200_ids.txt`: 25 per VSI-Bench task, relative
direction 8 easy / 8 medium / 9 hard, from 25 scenes), so its direct-answer predictions can be reused. Every condition:
Qwen3.5-2B, thinking off, 16 frames (`uniform128`), the sheet as csv, greedy decoding, up to 2,048 new tokens, up to 8
questions of one video per batch, and one rule for reading the answer (`score_final`):

| Condition | README | Sheet | Reasoning |
|---|---|---|---|
| `A_video` | A | none | direct |
| `video_cot` | B | none | structured CoT |
| `gold_csv` | C | gold | direct |
| `gold_cot` | D | gold | structured CoT |
| `gold_relations` | E | gold + closest-point relations | direct |

- **Structured CoT** is the eval's `--reasoning cot`, whose text is `cot_prompt.md` (checked below): say which objects
  the question is about and find them in the video and the sheet, write down their positions, distances or sizes, work
  the answer out one step at a time, and end on "Answer: X".
- **The direct conditions are copied** from explore200's predictions on Drive (`ssf_explore200_results/`: `A_video`,
  `gold_csv`, and `gold_relations_v2` for `gold_relations`, the same relations text) when they answer these 200
  questions with the prompts this code builds: the same frames and, batch by batch, the same input length. Otherwise
  they run here, a few minutes each.
- **Token limit 2,048**, as the structured-CoT plan and explore200's runs: on the 20 smoke questions
  (`results/_scratch/scot_smoke12/`, run locally with 2,048) the median CoT reply is 413 tokens without a sheet and 480
  with the gold sheet, and a limit of 256 would cut off 19 and 17 of the 20 replies.
- **Time:** the eval puts the questions of one video in a batch, and a batch runs until its longest reply ends, so the
  time grows with the number of batches (37 per condition here), not of questions. One batch of 5 questions is timed
  per CoT condition first; the notebook then prints an estimate and, above `HOURS_LIMIT` (2 h), stops until you choose
  `SUBSET`.
- **Comparisons** (Summary): `gold_cot − gold_csv` (does CoT help with the sheet), `video_cot − A_video` (without
  it), `gold_relations − gold_cot` (do precomputed relations beat computing them in writing), with bootstrap 95%
  intervals.

**After a crash**, run the notebook again from the top: data already in this runtime is not fetched again, a condition
whose predictions on Drive answer every question is skipped without loading the model, a partial one continues from
its last saved batch, and the timing is kept (`timing.json`).

**Files on Google Drive**, in `MyDrive/mvp/`: `mvp-3d-vlm-cot.zip`, the code, made in the repository (committed or
not) with

```
REV=$(git stash create); git archive --format=zip -o mvp-3d-vlm-cot.zip ${REV:-HEAD} src experiments/2026-11-18-scene-sheet-formats experiments/2026-11-19-structured-cot requirements.txt pyproject.toml
```

the scene-sheets zip (`SHEETS_ZIP`) and, to reuse them, explore200's predictions in `ssf_explore200_results/`.
VSI-Bench's `test.jsonl` and the 25 videos come from Hugging Face. Predictions go to
`MyDrive/mvp/scot_explore_results/`.

## Paths

In [ ]:
USE_DRIVE = True                     # False: put the files in /content yourself (Files pane) and set DRIVE_DIR to "/content"
DRIVE_DIR = "/content/drive/MyDrive/mvp"
CODE_ZIP = f"{DRIVE_DIR}/mvp-3d-vlm-cot.zip"  # the code (the first cell says how to make it)
DATA_SOURCE = "huggingface"          # "huggingface": VSI-Bench from Hugging Face + SHEETS_ZIP | "bundle": DATA_TAR from the cluster
SHEETS_ZIP = f"{DRIVE_DIR}/scene_sheets_75.zip"  # the scene-sheets zip: set to its name on your Drive
DATA_TAR = f"{DRIVE_DIR}/ssf_explore_data.tar.gz"  # only for DATA_SOURCE = "bundle"
WORK = "/content/mvp-3d-vlm"         # the code and data are unpacked here
RESULTS_DIR = f"{DRIVE_DIR}/scot_explore_results" if USE_DRIVE else "/content/scot_explore_results"  # this notebook's predictions (on Drive, they survive a disconnect and the runs resume)
REUSE_DIR = f"{DRIVE_DIR}/ssf_explore200_results"  # explore200.ipynb's predictions: the direct conditions come from here if they check out

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Every condition has the same model, frames, token limit, batches and answer rule (COMMON); they differ only in the
# sheet and the reasoning instructions. name -> (sheet source or None, reasoning), with the README's condition.
MODEL = "Qwen/Qwen3.5-2B"            # thinking off: the eval turns it off for Qwen3.5
EXP = f"{WORK}/experiments/2026-11-19-structured-cot"
SSF = f"{WORK}/experiments/2026-11-18-scene-sheet-formats"  # the format experiment: the 200 questions, cluster_checks.py
IDS_PATH = f"{SSF}/explore200_ids.txt"
SHEETS_ROOT = f"{WORK}/data/scene_sheets_75"
MAX_NEW_TOKENS = 2048                # every condition, as explore200's runs and the structured-CoT plan
BATCH_SIZE = 8                       # the eval's default, as explore200's runs: up to 8 questions of one video at a time
TOKEN_BUDGET = 120000                # the eval's default --token-budget (input tokens per batch), given for the estimate
COMMON = ["--model", MODEL, "--nframes", "16", "--sampling", "uniform128", "--max-new-tokens", str(MAX_NEW_TOKENS),
          "--final-answer", "--batch-size", str(BATCH_SIZE), "--token-budget", str(TOKEN_BUDGET)]
CONDITIONS = {
    "A_video": (None, "direct"),                     # A: video only
    "gold_csv": ("gold", "direct"),                  # C: video + gold sheet
    "gold_relations": ("gold_relations", "direct"),  # E: video + gold sheet + closest-point relations
    "video_cot": (None, "cot"),                      # B: video only, structured CoT
    "gold_cot": ("gold", "cot"),                     # D: video + gold sheet, structured CoT
}
DIRECT = [n for n, (_, r) in CONDITIONS.items() if r == "direct"]
COT = [n for n, (_, r) in CONDITIONS.items() if r == "cot"]
COMPARISONS = [("gold_cot", "gold_csv"), ("video_cot", "A_video"), ("gold_relations", "gold_cot")]  # first minus second
# explore200's files for the direct conditions. Its gold_relations_v2 has the closest-point relations, the same text as
# --sheet-source gold_relations writes; its "gold_relations" was centre-to-centre and is not used.
REUSE = {"A_video": "A_video.jsonl", "gold_csv": "gold_csv.jsonl", "gold_relations": "gold_relations_v2.jsonl"}
REUSE_DIRECT = True                  # False: run the direct conditions here instead (a few minutes each)
TIMING_N = 5                         # questions timed per CoT condition before the full runs: one video, one batch
HOURS_LIMIT = 2                      # an estimate above this stops the notebook to let you choose SUBSET
SUBSET = None                        # None: all 200 if the CoT runs fit in HOURS_LIMIT, else stop and ask;
                                     # "all": all 200 anyway; an int k: k questions per question type (Runs)


def eval_args(name):
    """The eval's arguments for a condition, without --ids and --out."""
    source, reasoning = CONDITIONS[name]
    args = [*COMMON, "--reasoning", reasoning]
    if source:
        args += ["--sheet-source", source, "--sheet-format", "csv", "--sheets-root", SHEETS_ROOT]
    return args

## Setup

In [ ]:
import os, shutil, subprocess, sys, zipfile

os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(WORK)
    CODE_COMMIT = z.comment.decode() or "unknown"  # git archive writes the commit it archived as the zip's comment
print(f"code: {CODE_ZIP}, commit {CODE_COMMIT}")
# what this notebook needs from the code: reply lengths in the predictions, the processor's batch, the CoT prompt's text
needs = {"src/mvp/eval/vsibench.py": "hit_token_limit", "src/mvp/models/qwen_vl.py": "def video_inputs",
         "experiments/2026-11-19-structured-cot/cot_prompt.md": "## Answer line",
         "experiments/2026-11-18-scene-sheet-formats/explore200_ids.txt": ""}
old = [f for f, text in needs.items() if not os.path.exists(f"{WORK}/{f}") or text not in open(f"{WORK}/{f}").read()]
assert not old, f"{CODE_ZIP} lacks what this notebook needs ({', '.join(old)}): make it again as the first cell says"

# requirements.txt pins torch 2.14.1+cu130; Colab's own torch is built for its driver, so it is kept unless this is False
KEEP_COLAB_TORCH = True
reqs = []
for line in open(f"{WORK}/requirements.txt").read().splitlines():
    line = line.split("#")[0].strip()
    if not line or (KEEP_COLAB_TORCH and (line.startswith("--extra-index-url") or line.split("==")[0] in ("torch", "torchvision"))):
        continue
    reqs.append(line)
# decord has no wheels for recent Python versions; eva-decord is a drop-in build imported under the same name
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[r for r in reqs if r != "decord"]], check=True)
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord"]).returncode != 0:
    print("decord has no wheel for this Python; installing eva-decord instead")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "eva-decord"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", WORK], check=True)
sys.path.insert(0, f"{WORK}/src")
print("installed:", ", ".join(reqs))

In [ ]:
import torch

print("torch", torch.__version__, "CUDA", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > GPU (L4 or A100)"
GPU = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f"GPU: {GPU}, compute capability {cap[0]}.{cap[1]}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")
if "T4" in GPU or cap < (8, 0):
    print("WARNING: this GPU has no native bfloat16 (T4 or older). The model runs in bfloat16, which is emulated here "
          "and several times slower; switch to an L4 or A100 if you can.")

## Data

Laid out as the code expects it: `data/raw/VSI-Bench/` (`test.jsonl`, `<dataset>/<scene>.mp4`),
`data/vggt/<dataset>/<scene>/` (`frame_indices.npy`, `meta.json`) and `data/scene_sheets_75/` (`index.csv`, the
questions' sheet folders, `adapter_report.json`).

In [ ]:
import csv, io, json, shutil, tarfile, zipfile
from pathlib import Path

import numpy as np

VSI_REPO = "nyu-visionx/VSI-Bench"
RAW, VGGT, SHEETS = Path(WORK) / "data/raw/VSI-Bench", Path(WORK) / "data/vggt", Path(SHEETS_ROOT)
IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]


def save(data, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(data)


def export_frames(video, max_frames=128):
    """frame_indices and fps exactly as the VGGT export takes them (mvp.reconstruction.extract.load_frames): decord's
    frame count and average fps, and min(128, N) frames spaced by np.linspace(0, N - 1, dtype=int)."""
    from decord import VideoReader, cpu
    vr = VideoReader(str(video), ctx=cpu(0))
    n = len(vr)
    idx = np.linspace(0, n - 1, num=min(max_frames, n), dtype=int)
    return idx, n, float(vr.get_avg_fps())


if DATA_SOURCE == "bundle":
    with tarfile.open(DATA_TAR) as t:
        t.extractall(WORK)
    print("unpacked", DATA_TAR)
else:
    from huggingface_hub import HfFileSystem, hf_hub_download

    # 1. the sheets: index.csv, each question's pipeline and gold sheet folders, each scene's adapter_report.json
    with zipfile.ZipFile(SHEETS_ZIP) as z:
        names = z.namelist()
        index_name = min((n for n in names if n.endswith("index.csv")), key=len)  # the top-level one
        prefix = index_name[: -len("index.csv")]
        rows = {int(r["id"]): r for r in csv.DictReader(io.TextIOWrapper(z.open(index_name), encoding="utf-8"))}
        absent = [q for q in IDS if q not in rows]
        assert not absent, f"{SHEETS_ZIP}: questions {absent} are not in its index.csv"
        scenes = sorted({rows[q]["scene"] for q in IDS})
        wanted = {rows[q][k] for q in IDS for k in ("pipeline_sheet", "gold_sheet")}
        wanted |= {f"{s.replace('/', '__')}/subset-uniform16/scene/adapter_report.json" for s in scenes}
        save(z.read(index_name), SHEETS / "index.csv")
        n_files = 0
        for name in names:
            parts = name[len(prefix):].split("/")
            if (name.startswith(prefix) and not name.endswith("/")
                    and any("/".join(parts[:i]) in wanted for i in range(1, len(parts) + 1))):
                save(z.read(name), SHEETS.joinpath(*parts))
                n_files += 1
        print(f"sheets: index.csv and {n_files} files: the pipeline and gold sheets of the {len(IDS)} questions, "
              f"{len(scenes)} scenes")
        # 2. frame_indices.npy and meta.json: from the zip if it has them for these scenes
        in_zip = {}
        for s in scenes:
            for f in ("frame_indices.npy", "meta.json"):
                hits = [n for n in names if n.endswith(f"/{s}/{f}") or n.endswith(f"/{s.replace('/', '__')}/{f}")]
                if hits:
                    in_zip[(s, f)] = min(hits, key=len)
        for s in scenes:
            if (s, "frame_indices.npy") in in_zip and (s, "meta.json") in in_zip:
                for f in ("frame_indices.npy", "meta.json"):
                    save(z.read(in_zip[(s, f)]), VGGT / s / f)
                print(f"{s}: frame_indices.npy and meta.json taken from the zip ({in_zip[(s, 'meta.json')]})")

    # 3. VSI-Bench's test.jsonl and the scenes' videos, read out of the per-dataset zips without downloading them whole
    RAW.mkdir(parents=True, exist_ok=True)
    if not (RAW / "test.jsonl").exists():  # after a crash in the same runtime, what is here is not fetched again
        shutil.copy(hf_hub_download(VSI_REPO, "test.jsonl", repo_type="dataset"), RAW / "test.jsonl")
    fs = HfFileSystem()
    for dataset in sorted({s.split("/")[0] for s in scenes if not (RAW / f"{s}.mp4").exists()}):
        with fs.open(f"datasets/{VSI_REPO}/{dataset}.zip", "rb", block_size=2**20) as f, zipfile.ZipFile(f) as z:
            for s in [s for s in scenes if s.startswith(dataset + "/") and not (RAW / f"{s}.mp4").exists()]:
                save(z.read(f"{s}.mp4"), RAW / f"{s}.mp4")
                print(f"{s}.mp4: {(RAW / f'{s}.mp4').stat().st_size / 2**20:.1f} MB from {dataset}.zip")

    # 4. recompute frame_indices.npy and meta.json where the zip had none
    for s in scenes:
        if (VGGT / s / "meta.json").exists() and (VGGT / s / "frame_indices.npy").exists():
            continue
        idx, n, fps = export_frames(RAW / f"{s}.mp4")
        (VGGT / s).mkdir(parents=True, exist_ok=True)
        np.save(VGGT / s / "frame_indices.npy", idx.astype(np.int32))
        (VGGT / s / "meta.json").write_text(json.dumps({
            "video": f"data/raw/VSI-Bench/{s}.mp4", "num_video_frames": n, "video_fps": fps, "num_frames": len(idx),
            "recomputed": "explore_cot.ipynb: frame_indices and video_fps as mvp.reconstruction.extract.load_frames "
                          "computes them; no VGGT outputs"}, indent=1))
        print(f"{s}: recomputed from the video: {n} frames, {fps} fps, {len(idx)} frame indices")

In [ ]:
# The video's "<t seconds>" labels come from the processor, from the fps the eval's video reader reports
# (qwen_vl_utils uses torchcodec, decord or torchvision, the first one installed). torchcodec measures the fps over the
# whole stream (frames / duration) and decord reads it from the header, and the two can differ in the last bits of the
# float (30.000000000000004 against 30.0), which flips a pair's label when its mean time is on a .x5 boundary. The sheets'
# timestamps (scene_sheet.frame_timestamps) use meta.json's video_fps, so it is set here to exactly what the eval's
# reader reports, read with the eval's own QwenVL.load_video; the export's value is kept as export_video_fps.
import inspect
import json
import sys

sys.path.insert(0, f"{WORK}/src")
from mvp.benchmarks import scene_sheet as ss
from mvp.models.qwen_vl import QwenVL
from qwen_vl_utils.vision_process import get_video_reader_backend

reader = QwenVL.__new__(QwenVL)  # load_video only needs QwenVL's pixel limits, not the model or the processor
limits = inspect.signature(QwenVL.__init__).parameters
reader.min_pixels, reader.max_pixels = limits["min_pixels"].default, limits["max_pixels"].default
backend = get_video_reader_backend()
index = ss.load_index(SHEETS)
changed = []
for s in sorted({index[q]["scene"] for q in IDS}):
    path = VGGT / s / "meta.json"
    meta = json.loads(path.read_text())
    _, video_meta, _ = reader.load_video(str(RAW / f"{s}.mp4"), nframes=ss.N_FRAMES)
    fps = float(video_meta["fps"])
    meta.setdefault("export_video_fps", meta["video_fps"])
    if meta["video_fps"] != fps or meta.get("video_reader_backend") != backend:
        if meta["video_fps"] != fps:
            changed.append(f"{s}: {meta['video_fps']!r} -> {fps!r}")
        meta.update(video_fps=fps, video_reader_backend=backend)
        path.write_text(json.dumps(meta, indent=1))
ss.frame_timestamps.cache_clear()  # timestamps computed before this cell used the old fps
print(f"video reader: {backend}; meta.json's video_fps changed to the reader's in {len(changed)} scenes"
      + "".join(f"\n  {c}" for c in changed))

In [ ]:
# Check 1: the 16 frames the eval picks (uniform128: 16 of the 128 frame_indices) are the frames the sheets were made on
# (adapter_report.json's source_video_frames). If not, the sheets describe other frames than the model sees.
import sys
sys.path.insert(0, f"{WORK}/src")
from mvp.benchmarks import scene_sheet as ss
from mvp.reconstruction.sa_sampling import uniform_subset

index = ss.load_index(SHEETS)
problems = []
for s in sorted({index[q]["scene"] for q in IDS}):
    idx = np.load(VGGT / s / "frame_indices.npy")
    report = json.loads((SHEETS / s.replace("/", "__") / "subset-uniform16" / "scene" / "adapter_report.json").read_text())
    picked = idx[uniform_subset(len(idx), 16)].tolist()
    fps = json.loads((VGGT / s / "meta.json").read_text())["video_fps"]
    print(f"{s}: {len(idx)} frame indices, fps {fps}; the eval's 16 frames {picked}")
    if report["selected_frames"] != uniform_subset(128, 16):
        problems.append(f"{s}: the sheets were made on frames {report['selected_frames']} of the 128, not uniform16")
    if picked != report["source_video_frames"]:
        problems.append(f"{s}: the eval would show video frames {picked}, but the sheets were made on "
                        f"{report['source_video_frames']}")
if problems:
    raise RuntimeError("The video frames don't match the sheets, so the sheets would describe other frames than the "
                       "model sees:\n  " + "\n  ".join(problems) +
                       "\nThe videos or the recomputed frame_indices.npy differ from the cluster's. Use the cluster's "
                       "data (DATA_SOURCE = \"bundle\") instead.")
print("OK: every scene's 16 frames are the ones its sheets were made on")

## Input check

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, f"{WORK}/src")
from decord import VideoReader
from mvp.benchmarks import scene_sheet as ss
from mvp.benchmarks.vsibench import load_vsibench
from mvp.paths import VGGT_DIR, VSI_DIR

IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]
index = ss.load_index(Path(SHEETS_ROOT))
vsi = load_vsibench("full").set_index("id")
need = [VSI_DIR / "test.jsonl", Path(SHEETS_ROOT) / "index.csv"]
for q in IDS:
    row = index[q]
    pipe = Path(SHEETS_ROOT) / row["pipeline_sheet"]
    need += [Path(vsi.loc[q, "video_path"]), VGGT_DIR / row["scene"] / "frame_indices.npy",
             VGGT_DIR / row["scene"] / "meta.json", pipe / "question.json",
             pipe.parents[1] / "scene" / "adapter_report.json", Path(SHEETS_ROOT) / row["gold_sheet"] / "objects.csv"]
missing = sorted({str(p) for p in need if not p.exists()})
assert not missing, "missing files:\n" + "\n".join(missing)
assert all(vsi.loc[q, "question"] == index[q]["question"] for q in IDS), "index.csv and test.jsonl disagree on a question"
print(f"all {len(set(map(str, need)))} files present for {len(IDS)} questions")

for scene in sorted({index[q]["scene"] for q in IDS}):
    video = Path(vsi[vsi["scene_name"] == scene.split("/")[1]]["video_path"].iloc[0])
    vr = VideoReader(str(video))
    fps = json.loads((VGGT_DIR / scene / "meta.json").read_text())["video_fps"]
    print(f"{scene}: {len(vr)} frames, {fps:.2f} fps, {len(vr) / fps:.0f} s, "
          f"{sum(index[q]['scene'] == scene for q in IDS)} questions")

## Prompts

The conditions must differ only in the sheet and the reasoning instructions. Check 2: the CoT instructions are
`cot_prompt.md`'s. Check 3: `gold_cot`'s prompts as the eval prints them (video labels, sheet, the whole prompt).
Check 4: question by question, each CoT condition is its direct one with only the last line changed, and
`gold_relations` is `gold_csv` with the relations after the sheet.

In [ ]:
# Check 2: the CoT instructions the eval writes (mvp.benchmarks.vsibench: COT_STEPS, COT_HINTS and COT_ANSWER, put
# together by cot_prompt) are word for word the text of cot_prompt.md, which README.md's design refers to.
import re
from mvp.benchmarks import vsibench as vb

md_text = Path(f"{EXP}/cot_prompt.md").read_text()
quote = lambda heading: re.search(r"^> (.*)$", md_text.split(heading, 1)[1], re.M).group(1)
problems = []
if quote("## Steps, without a sheet (B)") != vb.COT_STEPS.format(where=""):
    problems.append("the steps without a sheet (video_cot)")
if quote("## Steps, with the gold sheet (D)") != vb.COT_STEPS.format(where=" and in the scene information above"):
    problems.append("the steps with the gold sheet (gold_cot)")
if dict(re.findall(r"^- `(\w+)`: (.*)$", md_text, re.M)) != vb.COT_HINTS:
    problems.append("the hints per task")
if (dict(re.findall(r"^- (multiple choice|numeric): (.*)$", md_text, re.M))
        != {"multiple choice": vb.COT_ANSWER["mca"], "numeric": vb.COT_ANSWER["na"]}):
    problems.append("the answer lines")
assert not problems, "cot_prompt.md and the eval's CoT prompt differ in: " + ", ".join(problems)
print("OK: the eval's CoT instructions are cot_prompt.md's. With the gold sheet they read:\n")
print(vb.COT_STEPS.format(where=" and in the scene information above"), "<the task's hint>")
print(vb.COT_ANSWER["mca"], "/", vb.COT_ANSWER["na"])

In [ ]:
# Check 3: gold_cot's prompts as the eval builds them (--print-prompt: the processor and the videos, not the weights),
# checked by the format experiment's cluster_checks.check_prompts (the video's "<t seconds>" labels are the sheet's
# timestamps, and the gold sheet is in the prompt) and, whole, against build_sheet_prompt with reasoning="cot". (Its own
# whole-prompt check compares with the direct prompt, so that one is replaced by the CoT comparison.)
import os, subprocess
sys.path.insert(0, SSF)
import cluster_checks
from mvp.benchmarks import scene_sheet as ss
from mvp.paths import VGGT_DIR

cmd = [sys.executable, "-m", "mvp.eval.vsibench", *eval_args("gold_cot"), "--ids", IDS_PATH, "--print-prompt", str(len(IDS))]
printed = subprocess.run(cmd, cwd=WORK, env={**os.environ, "PYTHONPATH": f"{WORK}/src"}, capture_output=True, text=True)
assert printed.returncode == 0, printed.stdout[-3000:] + printed.stderr[-3000:]
problems = [p for p in cluster_checks.check_prompts(printed.stdout, Path(SHEETS_ROOT), VGGT_DIR, source="gold")
            if "build_sheet_prompt" not in p]
heads = list(cluster_checks.HEADER.finditer(printed.stdout))
for head, nxt in zip(heads, heads[1:] + [None]):
    q = int(head.group(1))
    block = printed.stdout[head.end():nxt.start() if nxt else len(printed.stdout)]
    tokens = cluster_checks.FRAME_TOKENS.search(block)
    expected = ss.build_sheet_prompt(q, "gold", "csv", Path(SHEETS_ROOT), VGGT_DIR, reasoning="cot",
                                     frame_tokens=tokens.group(1) if tokens else None)
    if block.strip("\n") != expected.strip("\n"):
        problems.append(f"question {q}: the prompt is not build_sheet_prompt's with reasoning='cot'")
if len(heads) != len(IDS):
    problems.append(f"{len(heads)} prompts printed, not {len(IDS)}")
if problems:
    raise RuntimeError("gold_cot's prompts are not as they should be:\n  " + "\n  ".join(problems))
print(f"OK: {len(heads)} gold_cot prompts; the video's labels are the sheets' timestamps, the gold sheet is in each, and "
      "each is build_sheet_prompt's with the CoT instructions")

In [ ]:
# Check 4: what the conditions differ in, question by question, in the text the eval puts after the video (its own
# mvp.eval.vsibench.prompt_parts, with the eval's defaults for everything this notebook doesn't set):
#   video_cot is A_video and gold_cot is gold_csv with only the last line, lmms-eval's "answer directly", replaced by the
#   CoT instructions (cot_prompt); A_video and video_cot carry no sheet; gold_relations is gold_csv with the relations
#   right after the sheet; and every condition's text is the one build_sheet_prompt writes for it.
import argparse
from mvp.benchmarks import vsibench as vb
from mvp.benchmarks.vsibench import load_vsibench
from mvp.eval import vsibench as ev

ORDER = load_vsibench("full")
ORDER = ORDER[ORDER["id"].isin(IDS)]  # the eval's order: test.jsonl's, questions grouped by video as it runs them
DOCS = {d["id"]: d for d in ORDER.to_dict("records")}
TYPE = {q: DOCS[q]["question_type"] for q in IDS}


def namespace(name):
    """The eval's arguments prompt_parts reads: this condition's sheet and reasoning, the eval's defaults otherwise."""
    source, reasoning = CONDITIONS[name]
    return argparse.Namespace(sheets=source is not None, sheet_source=source, sheet_format="csv" if source else None,
                              cameras=False, sheets_root=SHEETS_ROOT, vggt_root=str(VGGT_DIR), time_unit="seconds",
                              visibility="bridged", step_order="move-turn", object_directions="floor", metric_scale=None, layout="sit",
                              reasoning=reasoning)


TEXT = {name: {q: ev.prompt_parts(DOCS[q], namespace(name)) for q in IDS} for name in CONDITIONS}
problems, n_rel = [], 0
for q in IDS:
    d, post = DOCS[q], {name: TEXT[name][q][1] for name in CONDITIONS}
    if any(TEXT[name][q][0] for name in CONDITIONS):
        problems.append(f"question {q}: text before the video")
    for cot, direct, kind in (("video_cot", "A_video", None), ("gold_cot", "gold_csv", "gold")):
        body, last = post[direct].rsplit("\n", 1)
        if (last not in (vb.MCA_POST_PROMPT, vb.NA_POST_PROMPT)
                or post[cot] != body + "\n" + vb.cot_prompt(d, kind is not None, kind)):
            problems.append(f"question {q}: {cot} is not {direct} with the CoT instructions as its last line")
    sheet = ss.render_sheet(q, "gold", "csv", Path(SHEETS_ROOT), vggt_root=VGGT_DIR)
    rel = ss.relations(q, Path(SHEETS_ROOT))
    n_rel += bool(rel)
    if sheet not in post["gold_csv"] or post["gold_relations"] != post["gold_csv"].replace(sheet, sheet + ("\n" + rel if rel else ""), 1):
        problems.append(f"question {q}: gold_relations is not gold_csv with the relations after the sheet")
    if post["A_video"] != vb.build_prompt(d) or sheet in post["video_cot"]:
        problems.append(f"question {q}: the video-only text is not lmms-eval's, or carries a sheet")
    for name, (source, reasoning) in CONDITIONS.items():
        if post[name] not in ss.build_sheet_prompt(q, source, "csv", Path(SHEETS_ROOT), VGGT_DIR, reasoning=reasoning):
            problems.append(f"question {q}: {name}'s text is not build_sheet_prompt's")
if problems:
    raise RuntimeError("The conditions differ in more than the sheet and the instructions:\n  " + "\n  ".join(problems[:20]))
print(f"OK, for all {len(IDS)} questions: the CoT conditions are the direct ones with the CoT instructions as the last "
      f"line, and gold_relations is gold_csv plus relations ({n_rel} questions have them; the others name fewer than "
      "two classes the sheet has, and their gold_relations prompt is gold_csv's)")

In [ ]:
# One question's whole text input in every condition (build_sheet_prompt: chat template, the video's "<t seconds>"
# labels with one <|video_pad|> per frame pair, the sheet, the question, the instructions). Change Q to see another.
Q = next((q for q in IDS if TYPE[q] == "object_abs_distance" and ss.relations(q, Path(SHEETS_ROOT))), IDS[0])
for name, (source, reasoning) in CONDITIONS.items():
    print(f"===== {name} =====")
    print(ss.build_sheet_prompt(Q, source, "csv", Path(SHEETS_ROOT), VGGT_DIR, reasoning=reasoning))

## Input lengths

Every question's input length in every condition, as the eval's batches have it (the processor only, no weights).
They show what the sheets and the CoT instructions add, give the eval's batch size per video for the time estimate,
and decide whether explore200's direct predictions were made with these prompts (Runs).

In [ ]:
import pandas as pd
from mvp.models.qwen_vl import QwenVL

PROMPTER = QwenVL(MODEL, weights=False)  # the processor and the tokenizer, without the weights
LENGTHS = {name: {} for name in CONDITIONS}
VIDEO_OF, FRAMES, VIS_TOKENS = {}, {}, {}
for path, group in ORDER.groupby("video_path", sort=False):
    video, meta, vkw = PROMPTER.load_video_frames(path, ev.frame_indices(path, "uniform128", 16, str(VGGT_DIR)))
    VIS_TOKENS[path] = video.shape[0] * video.shape[2] * video.shape[3] // (2 * 32 * 32)  # as the eval counts them
    docs = group.to_dict("records")
    for d in docs:
        VIDEO_OF[d["id"]], FRAMES[d["id"]] = path, list(meta["frames_indices"])
    for name in CONDITIONS:
        parts = [TEXT[name][d["id"]] for d in docs]
        mask = PROMPTER.video_inputs(video, meta, vkw, [p for _, p in parts], pres=[q for q, _ in parts])["attention_mask"]
        LENGTHS[name].update({d["id"]: int(n) for d, n in zip(docs, mask.sum(1))})
lengths = pd.DataFrame(LENGTHS).loc[IDS]
print(f"input tokens per question ({len(IDS)} questions; the video is {min(VIS_TOKENS.values())} to "
      f"{max(VIS_TOKENS.values())} of them):")
display(lengths.describe().loc[["mean", "min", "max"]].round(0))

## Runs

First the direct conditions, copied from explore200 where its predictions check out, else run. Then one batch per CoT
condition is timed, and the estimate for the rest decides the questions: all 200, or with `SUBSET = k`, k questions of
every question type (all of a type that has fewer: relative direction has 8, 8 and 9), taken scene by scene in a random
order of the scenes (seed 0), since the time grows with the number of videos. The direct conditions always cover all
200; the Summary uses the questions every condition answered.

In [ ]:
import json, math, os, subprocess, sys, threading, time
from collections import Counter
from pathlib import Path

os.makedirs(RESULTS_DIR, exist_ok=True)


def out(name):
    """The condition's predictions."""
    return Path(RESULTS_DIR) / f"{name}.jsonl"


def answered(path):
    """The question ids a prediction file already answers. A crash can leave a half-written last line, which would stop
    the eval's own resume (it reads every line as JSON): unreadable lines are dropped from the file first."""
    path = Path(path)
    if not path.exists():
        return set()
    lines = path.read_text().splitlines()
    good = []
    for line in lines:
        try:
            good.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    if len(good) != len(lines):
        print(f"{path.name}: dropped {len(lines) - len(good)} unreadable line(s) left by a crash")
        path.write_text("".join(json.dumps(r) + "\n" for r in good))
    return {r["id"] for r in good}


def count(path, ids):
    """How many of ids a prediction file answers, reading it as it is written (for the progress lines)."""
    try:
        lines = Path(path).read_text().splitlines()
    except FileNotFoundError:
        return 0
    got = set()
    for line in lines:
        try:
            got.add(json.loads(line)["id"])
        except (json.JSONDecodeError, KeyError):
            pass
    return len(got & set(ids))


def run_eval(name, ids_path, out_path, every=300):
    """The eval for one condition on the questions of ids_path (it skips those out_path already answers), with its
    output and, every `every` seconds, how many are answered. Returns (seconds, the eval's output)."""
    ids = [int(x) for x in Path(ids_path).read_text().split()]
    cmd = [sys.executable, "-m", "mvp.eval.vsibench", *eval_args(name), "--ids", str(ids_path), "--out", str(out_path)]
    print(f"===== {name}: {' '.join(cmd[1:])}", flush=True)
    t0, stop = time.time(), threading.Event()

    def progress():
        while not stop.wait(every):
            print(f"  [{name}: {count(out_path, ids)} of {len(ids)} answered, {(time.time() - t0) / 60:.0f} min]", flush=True)

    threading.Thread(target=progress, daemon=True).start()
    proc = subprocess.Popen(cmd, cwd=WORK, env={**os.environ, "PYTHONPATH": f"{WORK}/src"}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    stop.set()
    assert proc.wait() == 0, f"{name} failed"
    print(f"===== {name} done in {time.time() - t0:.0f} s\n", flush=True)
    return time.time() - t0, "".join(lines)


def batch_sizes(name, ids):
    """The batches the eval will make of the questions of ids that `name` has not answered: the questions of one video
    together, at most min(BATCH_SIZE, TOKEN_BUDGET // (visual tokens + 200)) at a time (as mvp.eval.vsibench does)."""
    done = answered(out(name))
    left = Counter(VIDEO_OF[q] for q in ids if q not in done)
    sizes = []
    for video, n in left.items():
        bs = max(1, min(BATCH_SIZE, TOKEN_BUDGET // (VIS_TOKENS[video] + 200)))
        sizes += [bs] * (n // bs) + [n % bs] * (n % bs > 0)
    return sizes

In [ ]:
# The direct conditions. explore200's predictions stand in for a run here when they answer the same questions, on the
# same frames, scored by the final-answer rule, with replies shorter than the token limit, and with the prompts this
# code builds: the eval records each batch's padded input length (n_input_tokens), so in every batch the longest
# prompt as built here must have exactly that length. (explore200's gold_relations_v2 ran through explore_sheets.py,
# whose relations patch the repository's version no longer applies to this eval; the length check shows whether those
# prompts had the relations.) Copied predictions are listed in reused.json.
def reuse_problems(name, path):
    """Why explore200's predictions in `path` cannot stand in for condition `name` (empty: they can)."""
    run = pd.read_json(path, lines=True)
    if len(run) != len(IDS) or set(run["id"]) != set(IDS):
        return [f"{run['id'].nunique()} questions in {len(run)} lines, not these {len(IDS)}"]
    problems = []
    if "score_first" not in run:
        problems.append("not scored with --final-answer")
    other = [q for q, f in zip(run["id"], run["frame_indices"]) if list(f) != FRAMES[q]]
    if other:
        problems.append(f"{len(other)} questions on other frames than the eval picks here")
    bad = []
    for (_, n), g in run.assign(video=run["id"].map(VIDEO_OF)).groupby(["video", "n_input_tokens"]):
        longest = max(LENGTHS[name][q] for q in g["id"])
        if longest != n:
            bad.append(f"questions {sorted(g['id'])}: {n} input tokens there, {longest} here")
    if bad:
        problems.append(f"{len(bad)} batches had other prompts, e.g. " + "; ".join(bad[:2]))
    longest_reply = max(map(len, PROMPTER.processor.tokenizer(list(run["prediction"].astype(str)),
                                                               add_special_tokens=False)["input_ids"]))
    if longest_reply >= MAX_NEW_TOKENS:
        problems.append(f"a reply of {longest_reply} tokens, which may have stopped at another limit")
    return problems


REUSED_JSON = Path(RESULTS_DIR) / "reused.json"
reused = json.loads(REUSED_JSON.read_text()) if REUSED_JSON.exists() else {}
plan = []
for name in DIRECT:
    have, src = answered(out(name)) & set(IDS), Path(REUSE_DIR) / REUSE[name]
    if len(have) == len(IDS):
        plan.append((name, "done", f"all {len(IDS)} answered" + (f", copied from {reused[name]}" if name in reused else "")))
    elif have:
        plan.append((name, "resume", f"{len(have)} of {len(IDS)} answered here"))
    elif not REUSE_DIRECT:
        plan.append((name, "run", "REUSE_DIRECT is False"))
    elif not src.exists():
        plan.append((name, "run", f"no {src}"))
    else:
        problems = reuse_problems(name, src)
        plan.append((name, "run", "; ".join(problems)) if problems else (name, "copy", f"{src}: same questions, frames and prompts"))
display(pd.DataFrame(plan, columns=["condition", "action", "why"]).set_index("condition"))

for name, action, _ in plan:
    if action == "done":
        continue
    if action == "copy":
        shutil.copy(Path(REUSE_DIR) / REUSE[name], out(name))
        reused[name] = str(Path(REUSE_DIR) / REUSE[name])
        print(f"===== {name}: copied {reused[name]}")
    else:
        reused.pop(name, None)
        run_eval(name, IDS_PATH, out(name))
    REUSED_JSON.write_text(json.dumps(reused, indent=1))

In [ ]:
# Time one batch per CoT condition: the TIMING_N questions of one video, the video with exactly TIMING_N questions and
# the most question types (the first in the eval's order among equals), so that the batch is the one the full run would
# make. They run into a scratch file; their answers are then added to the condition's predictions, so the full run
# doesn't redo them, and timing.json keeps the times, so a resumed notebook doesn't time again on the same GPU.
import re

BY_VIDEO = ORDER.groupby("video_path", sort=False)["id"].apply(list)
candidates = ([v for v, qs in BY_VIDEO.items() if len(qs) == TIMING_N]
              or [v for v, qs in BY_VIDEO.items() if len(qs) > TIMING_N] or list(BY_VIDEO.index))
TIMING_VIDEO = max(candidates, key=lambda v: len({TYPE[q] for q in BY_VIDEO[v][:TIMING_N]}))
TIMING_IDS = BY_VIDEO[TIMING_VIDEO][:TIMING_N]
TIMING_IDS_PATH = f"{EXP}/timing_ids.txt"
Path(TIMING_IDS_PATH).write_text("".join(f"{q}\n" for q in TIMING_IDS))
print(f"timed: {TIMING_VIDEO.split('VSI-Bench/')[-1]}, questions {TIMING_IDS} ({', '.join(TYPE[q] for q in TIMING_IDS)})")

TIMING_JSON = Path(RESULTS_DIR) / "timing.json"
timing = json.loads(TIMING_JSON.read_text()) if TIMING_JSON.exists() else {}
SETUP = {"gpu": GPU, "max_new_tokens": MAX_NEW_TOKENS, "batch_size": BATCH_SIZE, "questions": TIMING_IDS}
TIME_LINE = re.compile(r"\[1 videos, (\d+) qs, (\d+)s\]")  # the eval's progress line after its first video
for name in COT:
    if answered(out(name)) >= set(IDS):
        print(f"{name}: all {len(IDS)} questions answered, nothing to time")
        continue
    if "steps" in timing.get(name, {}) and all(timing[name].get(k) == v for k, v in SETUP.items()):
        continue  # timed before, with these settings on this GPU
    scratch = Path(RESULTS_DIR) / "timing" / f"{name}.jsonl"
    scratch.parent.mkdir(exist_ok=True)
    scratch.unlink(missing_ok=True)
    wall, log = run_eval(name, TIMING_IDS_PATH, scratch)
    m = TIME_LINE.search(log)
    assert m and int(m.group(1)) == len(TIMING_IDS), "the eval did not print its time for the timed video"
    recs = {r["id"]: r for r in map(json.loads, scratch.read_text().splitlines())}
    tokens = [recs[q]["n_new_tokens"] for q in TIMING_IDS]
    bs = max(1, min(BATCH_SIZE, TOKEN_BUDGET // (VIS_TOKENS[TIMING_VIDEO] + 200)))  # one batch unless BATCH_SIZE < TIMING_N
    timing[name] = {**SETUP, "batch_s": float(m.group(2)), "wall_s": round(wall, 1), "reply_tokens": tokens,
                    "hit_token_limit": [recs[q]["hit_token_limit"] for q in TIMING_IDS],
                    "batches": math.ceil(len(tokens) / bs),  # and the decoding steps they took: each batch's longest reply
                    "steps": sum(max(tokens[i:i + bs]) for i in range(0, len(tokens), bs))}
    have = answered(out(name))
    with open(out(name), "a") as f:
        f.writelines(json.dumps(r) + "\n" for q, r in recs.items() if q not in have)
    TIMING_JSON.write_text(json.dumps(timing, indent=1))
for name, t in timing.items():
    print(f"{name}: {len(t['questions'])} questions in {t['batches']} batch(es), {t['batch_s']:.0f} s on {t['gpu']} "
          f"(plus {t['wall_s'] - t['batch_s']:.0f} s to start the eval): {1000 * t['batch_s'] / t['steps']:.0f} ms a "
          f"decoding step; replies of {t['reply_tokens']} tokens, {sum(t['hit_token_limit'])} cut off at "
          f"{t['max_new_tokens']}")

In [ ]:
# The estimate per CoT condition: a decoding step takes the timed batch's time over its longest reply (its prefill
# counted in, so a little high), each batch left runs as many steps as its longest reply, expected from the timed
# replies' lengths, and each run starts the eval once. "If every batch hit the limit" is the upper bound.
import random


def expected_longest(lengths, b):
    """The expected longest of b replies drawn at random, with replacement, from `lengths`."""
    L, n = sorted(lengths), len(lengths)
    return sum(L[k] * (((k + 1) / n) ** b - (k / n) ** b) for k in range(n))


def estimate(name, ids):
    """(expected seconds, seconds if every batch hit the limit, batches) for `name` on the questions of ids it has
    not answered."""
    sizes = batch_sizes(name, ids)
    if not sizes:
        return 0.0, 0.0, 0
    t = timing[name]
    step = t["batch_s"] / max(t["steps"], 1)
    start = min([u["wall_s"] - u["batch_s"] for u in timing.values() if u["gpu"] == GPU] or [t["wall_s"] - t["batch_s"]])
    return (start + step * sum(expected_longest(t["reply_tokens"], b) for b in sizes),
            start + step * MAX_NEW_TOKENS * len(sizes), len(sizes))


def per_type(k, seed=0):
    """k questions of every question type (all of a type that has fewer), taken scene by scene in a random order of
    the scenes, so that they come from few videos."""
    videos = list(BY_VIDEO.index)
    random.Random(seed).shuffle(videos)
    need = Counter({t: min(k, n) for t, n in Counter(TYPE.values()).items()})
    chosen = []
    for v in videos:
        for q in BY_VIDEO[v]:
            if need[TYPE[q]] > 0:
                chosen.append(q)
                need[TYPE[q]] -= 1
        if not +need:
            break
    return sorted(chosen)


if SUBSET not in (None, "all") and not isinstance(SUBSET, int):
    raise ValueError(f"SUBSET must be None, 'all' or a number of questions per question type, not {SUBSET!r}")
options = {"all": IDS, **{k: per_type(k) for k in sorted({10, 5} | ({SUBSET} if isinstance(SUBSET, int) else set()), reverse=True)}}
rows = []
for key, ids in options.items():
    est = {c: estimate(c, ids) for c in COT}
    rows.append({"SUBSET": key, "questions": len(ids), "videos": len({VIDEO_OF[q] for q in ids}),
                 "CoT batches left": sum(e[2] for e in est.values()),
                 **{f"{c} (min)": round(e[0] / 60) for c, e in est.items()},
                 "CoT runs left (h)": round(sum(e[0] for e in est.values()) / 3600, 1),
                 "if every batch hit the limit (h)": round(sum(e[1] for e in est.values()) / 3600, 1)})
print(f"Estimated time left for the CoT runs on this GPU ({GPU}):")
display(pd.DataFrame(rows).set_index("SUBSET"))
left_all = sum(estimate(c, IDS)[0] for c in COT)
if SUBSET is None and left_all > HOURS_LIMIT * 3600:
    raise RuntimeError(
        f"The CoT runs on all {len(IDS)} questions would take about {left_all / 3600:.1f} h more on this GPU, over "
        f"HOURS_LIMIT = {HOURS_LIMIT} h. Choose in the settings cell (Paths): SUBSET = 10 or SUBSET = 5 (questions per "
        "question type, as in the table above), or SUBSET = \"all\" to run all of them anyway; then run the notebook "
        "again from the top. Nothing done so far is redone, and the timing is kept.")
CHOICE = "all" if SUBSET in (None, "all") else SUBSET
RUN_IDS = options[CHOICE]
RUN_IDS_PATH = f"{EXP}/cot_ids.txt"
Path(RUN_IDS_PATH).write_text("".join(f"{q}\n" for q in RUN_IDS))
left = sum(estimate(c, RUN_IDS)[0] for c in COT)
print(f"The CoT runs: {len(RUN_IDS)} questions (SUBSET = {CHOICE!r}), about {left / 60:.0f} min left; per question "
      f"type: {dict(Counter(TYPE[q] for q in RUN_IDS))}")

In [ ]:
for name in COT:
    done = answered(out(name)) & set(RUN_IDS)
    if len(done) == len(RUN_IDS):
        print(f"===== {name}: all {len(RUN_IDS)} questions answered, skipped", flush=True)
        continue
    if done:
        print(f"===== {name}: resuming, {len(done)} of {len(RUN_IDS)} answered", flush=True)
    run_eval(name, RUN_IDS_PATH, out(name))

## Answers: one rule, parse rate, cut-off replies

Every reply is read by one rule, the eval's final-answer rule (`score_final`), applied again here to the reused
predictions too: for multiple choice, the letter of the last "Answer: X" or "answer is X", else the last letter that
stands alone in the reply's last line; for numbers, the last number in the reply. A reply it reads nothing from scores 0.
A reply cut off at the token limit has no answer line, but the rule may still read a letter or number from its
unfinished reasoning: those are counted apart, and "score, cut-off replies wrong" counts them as 0.

In [ ]:
import random
import re

import numpy as np
from mvp.benchmarks.vsibench import MCA_QUESTION_TYPES, TASKS, final_letter, final_number, parsed, score_final, summarize, task_of

RUN_IDS = globals().get("RUN_IDS", IDS)
ANSWER_LINE = re.compile(r"^[\s*#>_-]*answer\s*\**\s*:", re.I)


def read(d, reply):
    """The answer the rule reads from a reply: a letter, a number, or None."""
    return final_letter(reply) if d["question_type"] in MCA_QUESTION_TYPES else final_number(reply)


def answer_line(reply):
    """The reply's last line that starts with "Answer:", or None."""
    lines = [line for line in reply.splitlines() if ANSWER_LINE.match(line)]
    return lines[-1] if lines else None


def same(a, b):
    if isinstance(a, str) or isinstance(b, str):
        return a == b
    return abs(float(a) - float(b)) < 1e-9


runs = {}
for name in CONDITIONS:
    if not out(name).exists():
        continue
    run = pd.read_json(out(name), lines=True).drop_duplicates("id").set_index("id")
    run = run.loc[[q for q in RUN_IDS if q in run.index]].copy()
    run["prediction"] = run["prediction"].astype(str)
    docs = [DOCS[q] for q in run.index]
    run["rescored"] = [score_final(d, p) for d, p in zip(docs, run["prediction"])]
    run["read"] = pd.Series([read(d, p) for d, p in zip(docs, run["prediction"])], index=run.index, dtype=object)
    run["parsed"] = [parsed(d, p, final=True) for d, p in zip(docs, run["prediction"])]
    run["line"] = pd.Series([answer_line(p) for p in run["prediction"]], index=run.index, dtype=object)
    run["line_read"] = pd.Series([read(d, l) if l else None for d, l in zip(docs, run["line"])], index=run.index, dtype=object)
    # reply lengths: the eval's count; explore200's reused replies predate it and are counted with the tokenizer
    for col in ("n_new_tokens", "hit_token_limit"):
        if col not in run:
            run[col] = None
    missing = run["n_new_tokens"].isna()
    if missing.any():
        counted = [len(t) for t in PROMPTER.processor.tokenizer(list(run.loc[missing, "prediction"]),
                                                                add_special_tokens=False)["input_ids"]]
        run.loc[missing, "n_new_tokens"] = counted
        run.loc[missing, "hit_token_limit"] = [n >= MAX_NEW_TOKENS for n in counted]
    run["n_new_tokens"] = run["n_new_tokens"].astype(int)
    run["hit_token_limit"] = run["hit_token_limit"].astype(bool)
    runs[name] = run
COMMON_IDS = [q for q in RUN_IDS if all(q in r.index for r in runs.values())]
print(f"conditions with predictions: {', '.join(runs)}; {len(COMMON_IDS)} of the {len(RUN_IDS)} questions answered in "
      f"all of them" + ("" if set(runs) == set(CONDITIONS) else f"; missing: {', '.join(set(CONDITIONS) - set(runs))}"))
differ = {n: int((abs(r["rescored"] - r["score"]) > 1e-9).sum()) for n, r in runs.items()}
print("scores here that differ from the eval's own (the same rule, so none should):", differ)

answers = {}
for name, run in runs.items():
    cut, has_line = run["hit_token_limit"], run["line"].notna()
    answers[name] = {
        "replies": len(run),
        "parsed %": 100 * run["parsed"].mean(),
        "'Answer:' line %": 100 * has_line.mean(),
        f"hit {MAX_NEW_TOKENS} tokens": int(cut.sum()),
        "of them before answering": int((cut & ~has_line).sum()),
        "of those, read anyway": int((cut & ~has_line & run["parsed"]).sum()),
        "rule not the 'Answer:' line": sum(a is not None and b is not None and not same(a, b)
                                           for a, b in zip(run["read"], run["line_read"])),
        "median tokens": run["n_new_tokens"].median(),
        "90th pct tokens": run["n_new_tokens"].quantile(0.9),
        "mean score": 100 * run["rescored"].mean(),
        "score, cut-off replies wrong": 100 * run["rescored"].where(~(cut & ~has_line), 0).mean(),
    }
answers = pd.DataFrame.from_dict(answers, orient="index")
display(answers.round(1))

In [ ]:
# The CoT replies per question type: median length, how many hit the limit, and the parse rate.
per_type_cot = pd.concat({n: runs[n].groupby("question_type").agg(
    **{"median tokens": ("n_new_tokens", "median"), f"hit {MAX_NEW_TOKENS}": ("hit_token_limit", "sum"),
       "parsed %": ("parsed", lambda s: 100 * s.mean()), "n": ("parsed", "size")}) for n in COT if n in runs}, axis=1)
display(per_type_cot.round(0))

## Per-question table

In [ ]:
# Each condition's answer as the rule reads it, ✓/✗ (multiple choice) or the score (numbers), ✂ if cut off at the limit.
def cell(run, q):
    if q not in run.index:
        return "—"
    a, s = run.loc[q, "read"], run.loc[q, "rescored"]
    mark = " ✂" if run.loc[q, "hit_token_limit"] else ""
    if TYPE[q] in MCA_QUESTION_TYPES:
        return f"{a} {'✓' if s == 1 else '✗'}{mark}"
    return f"{'?' if pd.isna(a) else f'{a:g}'} ({s:.2f}){mark}"


table = pd.DataFrame({"scene": {q: DOCS[q]["dataset"] + "/" + DOCS[q]["scene_name"] for q in RUN_IDS},
                      "type": {q: TYPE[q] for q in RUN_IDS}, "truth": {q: DOCS[q]["ground_truth"] for q in RUN_IDS}})
for name, run in runs.items():
    table[name] = [cell(run, q) for q in table.index]
table.index.name = "id"
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 250)
display(table)

Full replies, for any question: `runs['gold_cot'].loc[QUESTION_ID, 'prediction']`.

## Summary

Score ×100 on the questions every condition answered: per question type, VSI-Bench's overall (the mean of the 8 task
scores, relative direction's three types as one task), and the overall without counting, whose answer the gold sheet
contains.

In [ ]:
TYPE_ORDER = [t for k in TASKS for t in ([f"object_rel_direction_{x}" for x in ("easy", "medium", "hard")]
                                         if k == "object_rel_direction" else [k])]
S = pd.DataFrame({n: r.loc[COMMON_IDS, "rescored"] for n, r in runs.items()})
types = pd.Series({q: TYPE[q] for q in COMMON_IDS})
qtypes = [t for t in TYPE_ORDER if t in set(types)]
by_type = (100 * S.groupby(types).mean()).reindex(qtypes)
for label, keep in (("overall (8 tasks)", types.index), ("overall without counting", types.index[types != "object_counting"])):
    by_type.loc[label] = [summarize(pd.DataFrame({"question_type": types[keep], "score": S.loc[keep, n]}))["overall"]
                          for n in S]
by_type.insert(0, "n", types.value_counts().reindex(by_type.index).fillna(len(types)).astype(int))
by_type.loc["overall without counting", "n"] = int((types != "object_counting").sum())
display(by_type.round(1))

## Bootstrap intervals

The three comparisons, each as the difference and its 95% interval from 10,000 paired resamples: questions are drawn
with replacement within each question type, so every type keeps its number of questions, and all conditions get the
same draw. `*`: the interval excludes 0. With 25 questions a type (fewer with `SUBSET`), a question type's interval is
wide; read the overall rows first.

In [ ]:
B = 10_000
rng = np.random.default_rng(0)
conds = list(S.columns)
X = S.to_numpy(float)
qtype_of = np.array([qtypes.index(t) for t in types])
members = [np.flatnonzero(qtype_of == i) for i in range(len(qtypes))]


def task_weights(drop=()):
    """w such that w @ (per question type means) is the mean of the task scores, without the tasks in drop."""
    tasks = [k for k in dict.fromkeys(task_of(t) for t in qtypes) if k not in drop]
    return np.mean([[(task_of(t) == k) / sum(task_of(u) == k for u in qtypes) for t in qtypes] for k in tasks], axis=0)


W = {"overall (8 tasks)": task_weights(), "overall without counting": task_weights({"object_counting"})}
point = np.stack([X[m].mean(0) for m in members])  # question type x condition
assert np.allclose(100 * W["overall (8 tasks)"] @ point, by_type.loc["overall (8 tasks)", conds].to_numpy(float))
boot = np.stack([np.stack([X[rng.choice(m, size=len(m))].mean(0) for m in members]) for _ in range(B)])


def interval(p, b):
    lo, hi = np.percentile(b, [2.5, 97.5])
    return f"{p:+.1f} [{lo:+.1f}, {hi:+.1f}]" + (" *" if lo > 0 or hi < 0 else "")


diffs, diff_numbers = {}, []
for x, y in COMPARISONS:
    if x not in conds or y not in conds:
        print(f"{x} − {y}: {x if x not in conds else y} has no predictions")
        continue
    i, j = conds.index(x), conds.index(y)
    d_point, d_boot = 100 * (point[:, i] - point[:, j]), 100 * (boot[:, :, i] - boot[:, :, j])
    rows = {**{label: (d_point @ w, d_boot @ w) for label, w in W.items()},
            **{t: (d_point[k], d_boot[:, k]) for k, t in enumerate(qtypes)}}
    diffs[f"{x} − {y}"] = {label: interval(p, b) for label, (p, b) in rows.items()}
    diff_numbers += [{"comparison": f"{x} - {y}", "rows": label, "difference": p,
                      "95% low": np.percentile(b, 2.5), "95% high": np.percentile(b, 97.5)} for label, (p, b) in rows.items()]
diffs = pd.DataFrame(diffs)
diffs.insert(0, "n", by_type["n"].reindex(diffs.index))
print(f"{len(COMMON_IDS)} questions answered in every condition; {B:,} stratified paired resamples")
display(diffs)

## Example CoT replies

One right, one wrong and one cut off at the token limit, drawn at random (`SEED`). The right and the wrong one come
from `gold_cot`, on the tasks that compute from the sheet's coordinates (absolute and relative distance, object size)
when there are any; the cut-off one from `gold_cot`, else `video_cot`. With the sheet the reply was given.

In [ ]:
SEED = 0
COMPUTING = {"object_abs_distance", "object_rel_distance", "object_size_estimation"}
rng_examples = random.Random(SEED)


def pick(kind):
    """(condition, question id) of a CoT reply of this kind: right (score 1), wrong (an answer read, not cut off, score
    0, else under 0.5) or cut (hit the limit, before an "Answer:" line if any did)."""
    for name in ("gold_cot", "video_cot"):
        if name not in runs:
            continue
        r = runs[name]
        cut = r["hit_token_limit"]
        if kind == "right":
            pool = list(r.index[(r["rescored"] == 1) & ~cut])
        elif kind == "wrong":  # scored 0, else (numbers) under 0.5
            pool = (list(r.index[(r["rescored"] == 0) & r["parsed"] & ~cut])
                    or list(r.index[(r["rescored"] < 0.5) & r["parsed"] & ~cut]))
        else:
            pool = list(r.index[cut & r["line"].isna()]) or list(r.index[cut])
        if kind != "cut":
            pool = [q for q in pool if TYPE[q] in COMPUTING] or pool
        if pool:
            return name, rng_examples.choice(sorted(pool))
    return None, None


def shorten(text, limit=6000):
    if len(text) <= limit:
        return text
    return text[:2 * limit // 3] + f"\n\n[... {len(text) - limit} characters left out ...]\n\n" + text[-limit // 3:]


for kind in ("right", "wrong", "cut"):
    name, q = pick(kind)
    print("=" * 100)
    if q is None:
        print(f"{kind}: no such CoT reply")
        continue
    r, d = runs[name].loc[q], DOCS[q]
    print(f"{kind.upper()}: {name}, question {q} ({TYPE[q]}, {d['dataset']}/{d['scene_name']})")
    print(d["question"])
    if TYPE[q] in MCA_QUESTION_TYPES:
        print("\n".join(d["options"]))
    print(f"ground truth: {d['ground_truth']}; the rule reads {r['read']!r}, score {r['rescored']:.2f}; "
          f"{r['n_new_tokens']} tokens" + (f", cut off at {MAX_NEW_TOKENS}" if r["hit_token_limit"] else ""))
    print("-" * 30, "reply")
    print(shorten(r["prediction"]))
    if CONDITIONS[name][0]:
        print("-" * 30, "the sheet it was given")
        print(ss.render_scene(q, CONDITIONS[name][0], "csv", root=Path(SHEETS_ROOT), vggt_root=VGGT_DIR))

## Save

In [ ]:
# The tables next to the predictions (on Drive when USE_DRIVE), and what this run was.
table.to_csv(f"{RESULTS_DIR}/per_question.csv")
answers.to_csv(f"{RESULTS_DIR}/answers.csv")
per_type_cot.to_csv(f"{RESULTS_DIR}/cot_replies_by_question_type.csv")
by_type.to_csv(f"{RESULTS_DIR}/by_question_type.csv")
pd.DataFrame(diff_numbers).to_csv(f"{RESULTS_DIR}/differences_bootstrap.csv", index=False)
Path(f"{RESULTS_DIR}/run_info.json").write_text(json.dumps({
    "code_zip": CODE_ZIP, "code_commit": CODE_COMMIT, "gpu": GPU, "model": MODEL, "common": COMMON,
    "conditions": CONDITIONS, "subset": globals().get("CHOICE"), "questions": len(COMMON_IDS),
    "reused": reused, "timing": timing}, indent=1))
print("saved to", RESULTS_DIR)